# Import Library dan Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import emoji
from sklearn.utils import resample
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('dataset.csv')
print(df.columns)
df.head()

Index(['review'], dtype='str')


,review
0,untuk pemesanan Shopee food bisa-bisanya drive...
1,"aplikasinya bagusss tapii,dulu dulu enak buat ..."
2,aplikasi yang rekomendasi sekali cocok untuk b...
3,ok
4,sangat meringankan untuk saat ini hanya kendal...


In [3]:
print(df.columns)

Index(['review'], dtype='str')


# Load lexicon

In [4]:
positive_df = pd.read_csv('positive.tsv', sep='\t')
negative_df = pd.read_csv('negative.tsv', sep='\t')

positive_words = set(positive_df['word'].astype(str))
negative_words = set(negative_df['word'].astype(str))

# Preprocessing

In [5]:
stemmer = StemmerFactory().create_stemmer()
stopwords_all = set([
    "yang","dan","di","ke","dari","ini","itu","untuk","dengan","pada",
    "saya","aku","kamu","dia","mereka","kami","kita",
    "ya","nya","aja","sih","kok","loh",
    "iya","yaa","na","kah","woi","woii","woy"
])

slangwords = {
    "gk": "tidak", "ga": "tidak", "nggak": "tidak",
    "bgt": "banget", "dr": "dari", "yg": "yang",
    "udh": "sudah", "tp": "tapi", "aja": "saja",
    "abis": "habis", "masi": "masih", "mantab": "mantap",
    "enk": "enak", "mudh": "mudah", "bgus": "bagus",
    "bgs": "bagus", "jelekkn": "jelek"
}

def normalize_repetition(word):
    return re.sub(r'(.)\1+', r'\1\1', word)

def tokenize(text):
    return re.findall(r'\b[a-zA-Z]+\b', text)

def preprocess(text):
    text = str(text)
    text = emoji.demojize(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    words = tokenize(text)
    words = [normalize_repetition(w) for w in words]
    words = [slangwords.get(w, w) for w in words]
    words = [w for w in words if w not in stopwords_all]
    words = [stemmer.stem(w) for w in words]
    return ' '.join(words)

df['clean'] = df['review'].apply(preprocess)
df[['review','clean']].head()

,review,clean
0,untuk pemesanan Shopee food bisa-bisanya drive...,mesan shopee food bisa bisa driver pick up ata...
1,"aplikasinya bagusss tapii,dulu dulu enak buat ...",aplikasi baguss tapi dulu dulu enak buat sen t...
2,aplikasi yang rekomendasi sekali cocok untuk b...,aplikasi rekomendasi sekali cocok belanja apa
3,ok,ok
4,sangat meringankan untuk saat ini hanya kendal...,sangat ringan saat hanya kendala kirim makan w...


In [6]:
def preprocess_lexicon(word):
    word = str(word)
    word = word.lower()
    word = normalize_repetition(word)
    word = slangwords.get(word, word)
    word = stemmer.stem(word)
    return word
positive_words = set([w for w in positive_words if " " not in w])
negative_words = set([w for w in negative_words if " " not in w])

positive_words = set([preprocess_lexicon(w) for w in positive_words])
negative_words = set([preprocess_lexicon(w) for w in negative_words])

# Labelling

In [ ]:
negation_words = set(["tidak","bukan","kurang","ga","gak","tak","tanpa","jangan"])
intensifier_words = set(["sangat","banget","sekali","terlalu","amat","paling"])

def label_sentiment(text):
    words = text.split()
    score = 0

    for i, w in enumerate(words):
        weight = 1

        if i > 0 and words[i-1] in intensifier_words:
            weight = 2

        is_negated = i > 0 and words[i-1] in negation_words

        if w in positive_words:
            score += -weight if is_negated else weight

        elif w in negative_words:
            score += weight if is_negated else -2 * weight  # KEMBALI ke -2*weight agar negatif terdeteksi lebih banyak

    if score > 0:
        return "positif"
    elif score < 0:
        return "negatif"
    else:
        return "netral"

df['sentiment'] = df['clean'].apply(label_sentiment)

sentiment
positif    7123
negatif    1717
netral     1160
Name: count, dtype: int64


In [71]:
print(df['sentiment'].value_counts())

sentiment
positif    7123
negatif    1717
netral     1160
Name: count, dtype: int64


In [ ]:
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    df['clean'], df['sentiment'],
    test_size=0.2, stratify=df['sentiment'], random_state=42
)

X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
    df['clean'], df['sentiment'],
    test_size=0.3, stratify=df['sentiment'], random_state=42
)

In [73]:
def balance_data(X_train_text, y_train):
    train_df = pd.DataFrame({'clean': X_train_text, 'sentiment': y_train})
    df_pos = train_df[train_df['sentiment'] == 'positif']
    df_neg = train_df[train_df['sentiment'] == 'negatif']
    df_net = train_df[train_df['sentiment'] == 'netral']

    max_len = len(df_pos)
    df_neg_up = resample(df_neg, replace=True, n_samples=max_len, random_state=42)
    df_net_up = resample(df_net, replace=True, n_samples=max_len, random_state=42)

    df_balanced = pd.concat([df_pos, df_neg_up, df_net_up]) \
                    .sample(frac=1, random_state=42).reset_index(drop=True)
    return df_balanced['clean'], df_balanced['sentiment']

X_train_80, y_train_80 = balance_data(X_train_80, y_train_80)
X_train_70, y_train_70 = balance_data(X_train_70, y_train_70)

print("Distribusi 80/20:", y_train_80.value_counts().to_dict())
print("Distribusi 70/30:", y_train_70.value_counts().to_dict())

Distribusi 80/20: {'negatif': 5698, 'netral': 5698, 'positif': 5698}
Distribusi 70/30: {'positif': 4986, 'netral': 4986, 'negatif': 4986}


Skema 1: Logistic Regression + TF-IDF

In [74]:
tfidf1 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                         min_df=2, max_df=0.95, sublinear_tf=True)
X_train1 = tfidf1.fit_transform(X_train_80)
X_test1  = tfidf1.transform(X_test_80)

model1 = LogisticRegression(max_iter=3000, C=1, class_weight='balanced', solver='lbfgs')
model1.fit(X_train1, y_train_80)

pred1 = model1.predict(X_test1)
print("Skema 1 - LR + TF-IDF (80/20)")
print("Accuracy:", accuracy_score(y_test_80, pred1))
print(classification_report(y_test_80, pred1))

Skema 1 - LR + TF-IDF (80/20)
Accuracy: 0.854
              precision    recall  f1-score   support

     negatif       0.72      0.82      0.76       343
      netral       0.58      0.57      0.58       232
     positif       0.94      0.91      0.92      1425

    accuracy                           0.85      2000
   macro avg       0.74      0.77      0.75      2000
weighted avg       0.86      0.85      0.86      2000



Skema 2: SVM + TF-IDF

In [75]:
tfidf2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                         min_df=2, max_df=0.95, sublinear_tf=True)
X_train2 = tfidf2.fit_transform(X_train_70)
X_test2  = tfidf2.transform(X_test_70)

model2 = LinearSVC(class_weight='balanced', C=0.5, max_iter=3000)
model2.fit(X_train2, y_train_70)

pred2 = model2.predict(X_test2)
print("Skema 2 - SVM + TF-IDF (70/30)")
print("Accuracy:", accuracy_score(y_test_70, pred2))
print(classification_report(y_test_70, pred2))

Skema 2 - SVM + TF-IDF (70/30)
Accuracy: 0.8636666666666667
              precision    recall  f1-score   support

     negatif       0.74      0.82      0.78       515
      netral       0.63      0.59      0.61       348
     positif       0.93      0.92      0.93      2137

    accuracy                           0.86      3000
   macro avg       0.77      0.78      0.77      3000
weighted avg       0.87      0.86      0.86      3000



Skema 3: RF + CountVectorizer

In [ ]:
cv3 = CountVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)
X_train3 = cv3.fit_transform(X_train_80)
X_test3  = cv3.transform(X_test_80)

model3 = LogisticRegression(
    max_iter=3000,
    C=1,
    class_weight='balanced',
    solver='lbfgs'
)
model3.fit(X_train3, y_train_80)

pred3 = model3.predict(X_test3)
print("Skema 3 - LR + CountVectorizer (80/20)")
print("Accuracy:", accuracy_score(y_test_80, pred3))
print(classification_report(y_test_80, pred3))

Skema 3 - LR + CountVectorizer (80/20)
Accuracy: 0.8505
              precision    recall  f1-score   support

     negatif       0.76      0.73      0.74       343
      netral       0.55      0.59      0.57       232
     positif       0.92      0.92      0.92      1425

    accuracy                           0.85      2000
   macro avg       0.74      0.75      0.75      2000
weighted avg       0.85      0.85      0.85      2000



Inference

In [130]:
def predict(text, model, vectorizer):
    text_clean = preprocess(text)
    vec = vectorizer.transform([text_clean])
    return model.predict(vec)[0]

texts = [
    "aplikasinya bagus banget, cepat dan mudah digunakan",       
    "jelek banget, error terus dan tidak bisa digunakan sama sekali", 
    "aplikasi ini cukup untuk kebutuhan sehari-hari"              
]

for t in texts:
    print(f"\nText: {t}")
    print("LR TF-IDF            :", predict(t, model1, tfidf1))
    print("SVM TF-IDF           :", predict(t, model2, tfidf2))
    print("LR CountVectorizer   :", predict(t, model3, cv3))


Text: aplikasinya bagus banget, cepat dan mudah digunakan
LR TF-IDF            : positif
SVM TF-IDF           : positif
LR CountVectorizer   : positif

Text: jelek banget, error terus dan tidak bisa digunakan sama sekali
LR TF-IDF            : negatif
SVM TF-IDF           : negatif
LR CountVectorizer   : negatif

Text: aplikasi ini cukup untuk kebutuhan sehari-hari
LR TF-IDF            : netral
SVM TF-IDF           : negatif
LR CountVectorizer   : netral
